# Pandas

[Vorlesungsaufzeichnung Daten putzen](https://media.hhu.de/channel/video/data-science-vl-9-stetige-verteilungen-2025/39016f5414ef3b8edd0dff941f802314/167?t=3244)

<!---
https://pbpython.com/categorical-encoding.html
--->

## DataFrame und Series

Der **DataFrame** ist die Datenstruktur, um die sich alles in Pandas dreht: eine Tabelle mit benannten Spalten, in der jede Spalte ihren eigenen Datentyp hat. Wer R kennt, kennt ihn als `data.frame`; wer bisher nur Numpy benutzt hat, kann ihn sich als record array vorstellen (`np.genfromtxt(..., names=True)` im Kapitel [Explorative Analyse](explorative-analyse-beispiele.ipynb) liefert so etwas), nur bequemer. Am ehesten ist es ein Excel-Blatt in Python.

Anders als ein Numpy-Array ist ein DataFrame immer zweidimensional — es gibt keine DataFrames mit drei oder vier Achsen. Das ist keine Einschränkung, sondern eine Festlegung auf genau die Form, in der Daten üblicherweise anfallen: Zeilen sind Beobachtungen, Spalten sind Merkmale.

In [1]:
import numpy as np
import pandas as pd

spalten = {
    "spezies": ["setosa", "setosa", "versicolor"],
    "kelchblattbreite": [3.5, 3.0, 3.2],
}
df = pd.DataFrame(spalten)
print(df)

      spezies  kelchblattbreite
0      setosa               3.5
1      setosa               3.0
2  versicolor               3.2


Auf eine Spalte greift man auf zwei Arten zu, die dasselbe tun: mit eckigen Klammern wie bei einem Dictionary, oder als Attribut. Die Klammern funktionieren immer, die Attributschreibweise nur, wenn der Spaltenname ein gültiger Python-Bezeichner ist und nicht zufällig eine Methode überdeckt — im Zweifel also Klammern.

In [2]:
print(df["kelchblattbreite"])
print(df.kelchblattbreite)  # dasselbe

print(df.dtypes)  # hier sieht man, dass Numpy dahintersteckt
print(df["kelchblattbreite"].to_numpy())

0    3.5
1    3.0
2    3.2
Name: kelchblattbreite, dtype: float64
0    3.5
1    3.0
2    3.2
Name: kelchblattbreite, dtype: float64
spezies                 str
kelchblattbreite    float64
dtype: object
[3.5 3.  3.2]


Eine einzelne Spalte ist kein DataFrame, sondern die zweite Datenstruktur von Pandas: eine **Series**, also ein eindimensionales Array mit Index. Die meisten Methoden gibt es für beide, manche nur für eine von beiden — die Fehlermeldung `'Series' object has no attribute ...` heißt meistens, dass man versehentlich eine Spalte statt der Tabelle erwischt hat.

Der Weg auf die Platte und zurück geht mit `to_csv` und `read_csv`; letzteres nimmt auch eine URL. Das `index=False` beim Schreiben ist wichtig, sonst landet der Zeilenindex als Spalte in der Datei und kommt beim Einlesen als `Unnamed: 0` zurück.

In [3]:
print(type(df["kelchblattbreite"]))

df.to_csv("beispiel.csv", index=False, encoding="utf-8")
wieder_da = pd.read_csv("beispiel.csv")
print(wieder_da)

import os
os.remove("beispiel.csv")

<class 'pandas.Series'>
      spezies  kelchblattbreite
0      setosa               3.5
1      setosa               3.0
2  versicolor               3.2


Pandas hat eine der besten Dokumentationen im Python-Ökosystem, und es lohnt sich, sie zu benutzen statt zu raten: die [Tutorials für den Einstieg](https://pandas.pydata.org/docs/getting_started/tutorials.html), die API-Referenz mit einer eigenen Seite je Methode (etwa zu `pandas.DataFrame.groupby`) und die [Übersicht der Datentypen](https://pandas.pydata.org/docs/reference/arrays.html).

Denn das Typensystem von Pandas geht über die Numpy-Typen hinaus. Neben `int64`, `float64` und `bool` gibt es unter anderem:

| Art der Daten | pandas-Datentyp | Skalar |
|---|---|---|
| Zeitpunkt mit Zeitzone | `DatetimeTZDtype` | `Timestamp` |
| Zeitdauer | — | `Timedelta` |
| Zeitraum | `PeriodDtype` | `Period` |
| Intervall | `IntervalDtype` | `Interval` |
| Ganzzahl mit fehlenden Werten | `Int64Dtype`, … | — |
| kategoriale Werte | `CategoricalDtype` | — |
| dünn besetzte Spalte | `SparseDtype` | — |
| Zeichenketten | `StringDtype` | `str` |
| Wahrheitswerte mit fehlenden Werten | `BooleanDtype` | — |

Zwei davon begegnen uns wieder: `CategoricalDtype` beim One-Hot-Encoding in der [Regressionsanalyse](lineare-regression.ipynb) und die Zeitstempel beim [Putzen der Daten](data-cleaning.ipynb). Dass die Spalte `spezies` oben den Typ `str` hat und nicht `object`, ist neu — bis pandas 2 waren Zeichenketten in einer `object`-Spalte untergebracht, was man in älterem Code und älteren Tutorials noch häufig sieht.

Aus der [Pandas Dokumentation, Abschnitt "Comparison with SQL"](https://pandas.pydata.org/docs/getting_started/comparison/comparison_with_sql.html).

In [4]:
import numpy as np
import pandas as pd

url = (
    "https://raw.github.com/pandas-dev"
    "/pandas/main/pandas/tests/io/data/csv/tips.csv"
)

tips = pd.read_csv(url)
print(tips)

     total_bill   tip     sex smoker   day    time  size
0         16.99  1.01  Female     No   Sun  Dinner     2
1         10.34  1.66    Male     No   Sun  Dinner     3
2         21.01  3.50    Male     No   Sun  Dinner     3
3         23.68  3.31    Male     No   Sun  Dinner     2
4         24.59  3.61  Female     No   Sun  Dinner     4
..          ...   ...     ...    ...   ...     ...   ...
239       29.03  5.92    Male     No   Sat  Dinner     3
240       27.18  2.00  Female    Yes   Sat  Dinner     2
241       22.67  2.00    Male    Yes   Sat  Dinner     2
242       17.82  1.75    Male     No   Sat  Dinner     2
243       18.78  3.00  Female     No  Thur  Dinner     2

[244 rows x 7 columns]


Operationen auf Pandas DataFrames erzeugen Kopien, während der ursprüngliche DataFrame unverändert bleibt.

In [5]:
tips.sort_values("total_bill")
print(tips)
print("\nDas ist ja gar nicht sortiert! Das nächste aber:\n")
sorted_tips = tips.sort_values("total_bill")
print(sorted_tips)

     total_bill   tip     sex smoker   day    time  size
0         16.99  1.01  Female     No   Sun  Dinner     2
1         10.34  1.66    Male     No   Sun  Dinner     3
2         21.01  3.50    Male     No   Sun  Dinner     3
3         23.68  3.31    Male     No   Sun  Dinner     2
4         24.59  3.61  Female     No   Sun  Dinner     4
..          ...   ...     ...    ...   ...     ...   ...
239       29.03  5.92    Male     No   Sat  Dinner     3
240       27.18  2.00  Female    Yes   Sat  Dinner     2
241       22.67  2.00    Male    Yes   Sat  Dinner     2
242       17.82  1.75    Male     No   Sat  Dinner     2
243       18.78  3.00  Female     No  Thur  Dinner     2

[244 rows x 7 columns]

Das ist ja gar nicht sortiert! Das nächste aber:

     total_bill    tip     sex smoker   day    time  size
67         3.07   1.00  Female    Yes   Sat  Dinner     1
92         5.75   1.00  Female    Yes   Fri  Dinner     2
111        7.25   1.00  Female     No   Sat  Dinner     1
172      

Spalten auswählen ist `SELECT`:

In [6]:
# SELECT total_bill, tip, smoker, time FROM tips;
tips[["total_bill", "tip", "smoker", "time"]]

,total_bill,tip,smoker,time
0,16.99,1.01,No,Dinner
1,10.34,1.66,No,Dinner
2,21.01,3.50,No,Dinner
3,23.68,3.31,No,Dinner
4,24.59,3.61,No,Dinner
...,...,...,...,...
239,29.03,5.92,No,Dinner
240,27.18,2.00,Yes,Dinner
241,22.67,2.00,Yes,Dinner
242,17.82,1.75,No,Dinner


Man kann neue Spalten hinzufügen, die sich aus alten Spalten berechnen:

In [7]:
# SELECT *, tip/total_bill as tip_rate FROM tips;
tips.assign(tip_rate=tips["tip"] / tips["total_bill"])

,total_bill,tip,sex,smoker,day,time,size,tip_rate
0,16.99,1.01,Female,No,Sun,Dinner,2,0.059447
1,10.34,1.66,Male,No,Sun,Dinner,3,0.160542
2,21.01,3.50,Male,No,Sun,Dinner,3,0.166587
3,23.68,3.31,Male,No,Sun,Dinner,2,0.139780
4,24.59,3.61,Female,No,Sun,Dinner,4,0.146808
...,...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3,0.203927
240,27.18,2.00,Female,Yes,Sat,Dinner,2,0.073584
241,22.67,2.00,Male,Yes,Sat,Dinner,2,0.088222
242,17.82,1.75,Male,No,Sat,Dinner,2,0.098204


Daten filtern (nach Bedingungen Zeilen auswählen) mit `WHERE`

In [8]:
# SELECT * FROM tips WHERE time = 'Dinner';
is_dinner = tips["time"] == "Dinner" # Bitmaske
tips[is_dinner]

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4
...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3
240,27.18,2.00,Female,Yes,Sat,Dinner,2
241,22.67,2.00,Male,Yes,Sat,Dinner,2
242,17.82,1.75,Male,No,Sat,Dinner,2


Mehrere Bedingungen logisch verknüpfen mit `OR |` oder `AND &`:

In [9]:
# SELECT * FROM tips WHERE time = 'Dinner' AND tip > 6.00;
tips[(tips["time"] == "Dinner") & (tips["tip"] > 6.00)]

,total_bill,tip,sex,smoker,day,time,size
23,39.42,7.58,Male,No,Sat,Dinner,4
59,48.27,6.73,Male,No,Sat,Dinner,4
170,50.81,10.00,Male,Yes,Sat,Dinner,3
183,23.17,6.50,Male,Yes,Sun,Dinner,4
212,48.33,9.00,Male,No,Sat,Dinner,4
214,28.17,6.50,Female,Yes,Sat,Dinner,3


In [10]:
# SELECT * FROM tips WHERE size >= 6 OR total_bill > 45;
tips[(tips["size"] >= 6) | (tips["total_bill"] > 45)]

,total_bill,tip,sex,smoker,day,time,size
59,48.27,6.73,Male,No,Sat,Dinner,4
125,29.80,4.20,Female,No,Thur,Lunch,6
141,34.30,6.70,Male,No,Thur,Lunch,6
143,27.05,5.00,Female,No,Thur,Lunch,6
156,48.17,5.00,Male,No,Sun,Dinner,6
170,50.81,10.00,Male,Yes,Sat,Dinner,3
182,45.35,3.50,Male,Yes,Sun,Dinner,3
212,48.33,9.00,Male,No,Sat,Dinner,4


Daten in Gruppen aufspalten, aggregierte Statistiken ausrechnen, Gruppen zusammenführen - das ist in SQL ein `GROUP BY`, in Pandas `groupby`:

In [11]:
# SELECT sex, count(*) FROM tips GROUP BY sex;
tips.groupby("sex").size()
# tips.groupby("sex").count() wäre falsch, das zählt nicht-NA-Einträge!

sex
Female     87
Male      157
dtype: int64

In [12]:
# SELECT day, AVG(tip), COUNT(*) FROM tips GROUP BY day;
tips.groupby("day").agg({"tip": "mean", "day": np.size})

,tip,day
day,,
Fri,2.734737,19
Sat,2.993103,87
Sun,3.255132,76
Thur,2.771452,62


Eine Falle, die beim Aggregieren immer wieder zuschnappt: **Der Mittelwert der Gruppenmittelwerte ist
nicht der Gesamtmittelwert**, außer alle Gruppen sind gleich groß. Der Gesamtmittelwert ist das
*gewichtete* Mittel der Gruppenmittel, mit den Gruppengrößen als Gewichten. Mit den Werten $10, 20, 30$
in Gruppe A und $5, 15$ in Gruppe B sind die Gruppenmittel $20$ und $10$, ihr Mittel ist $15$ — der
Gesamtmittelwert aller fünf Werte ist aber $16$. Wer eine Tabelle mit Gruppenmitteln vorliegen hat und
daraus das Gesamtmittel bilden will, braucht also zusätzlich die `size`-Spalte. Dasselbe Phänomen in
schärferer Form — Gruppenmittel, deren Ordnung sich beim Zusammenlegen umkehrt — heißt
*[Simpson-Paradoxon](deskriptive-statistik.md)*.


In [13]:
klein = pd.DataFrame({"Gruppe": ["A", "A", "A", "B", "B"], "Wert": [10, 20, 30, 5, 15]})
gruppenmittel = klein.groupby("Gruppe")["Wert"].mean()
print("Mittel der Gruppenmittel:", gruppenmittel.mean())
print("Gesamtmittel:            ", klein["Wert"].mean())
# richtig: mit den Gruppengroessen gewichten
groessen = klein.groupby("Gruppe")["Wert"].size()
print("gewichtet:               ", (gruppenmittel * groessen).sum() / groessen.sum())


Mittel der Gruppenmittel: 15.0
Gesamtmittel:             16.0
gewichtet:                16.0


In [14]:
# SELECT smoker, day, COUNT(*), AVG(tip) FROM tips GROUP BY smoker, day;
tips.groupby(["smoker", "day"]).agg({"tip": [np.size, "mean"]})

tip          
            size      mean
smoker day                
No     Fri     4  2.812500
       Sat    45  3.102889
       Sun    57  3.167895
       Thur   45  2.673778
Yes    Fri    15  2.714000
       Sat    42  2.875476
       Sun    19  3.516842
       Thur   17  3.030000

Daten kommen selten in einer einzigen Tabelle. Angenommen, die Iris-Messungen liegen in zwei Dateien: die eine enthält die Spezies jeder Pflanze, die andere ihre Kelchblattbreite. Beide haben gleich viele Zeilen — aber dürfen wir sie einfach nebeneinanderlegen? Nur, wenn die Zeilen in derselben Reihenfolge stehen, und das weiß man nicht, sobald irgendwer zwischendurch sortiert hat.

Deshalb bekommt jede Zeile eine Spalte, die sie eindeutig identifiziert: einen **Schlüssel**. Das kann eine schlichte Nummer sein, oder eine bereits vorhandene eindeutige Information (eine Matrikelnummer, eine ISBN). Wenn Daten aus mehreren Quellen zusammenkommen und niemand zentral Nummern vergeben kann, nimmt man eine **UUID** (*universally unique identifier*), eine 128-Bit-Zufallszahl, bei der ein Zusammenstoß so unwahrscheinlich ist, dass man ihn ignorieren darf. Die Spalte, die eine Tabelle eindeutig indiziert, heißt ihr **Primärschlüssel**; taucht derselbe Schlüssel in einer anderen Tabelle auf, um auf diese Zeilen zu verweisen, heißt er dort Fremdschlüssel. Das ist die Grundidee relationaler Datenbanken. Ausführlicher steht das im Kapitel [Relational data](https://r4ds.had.co.nz/relational-data.html) von *R for Data Science*.

In [15]:
import uuid

spezies = pd.DataFrame({"Nr": [1, 51], "spezies": ["setosa", "versicolor"]})
breite = pd.DataFrame({"Nr": [51, 1], "kelchblattbreite": [3.2, 3.5]})
print(spezies, "\n")
print(breite, "\n")

# ueber den Schluessel Nr zusammengefuehrt, Reihenfolge egal:
print(pd.merge(spezies, breite, on="Nr"), "\n")

print(uuid.uuid4())  # eine frische UUID

   Nr     spezies
0   1      setosa
1  51  versicolor 

   Nr  kelchblattbreite
0  51               3.2
1   1               3.5 

   Nr     spezies  kelchblattbreite
0   1      setosa               3.5
1  51  versicolor               3.2 

72d5277f-bf75-4083-8f88-e0d714f11f05


Interessant wird es, wenn die Schlüssel nicht in beiden Tabellen dieselben sind. Dann muss man entscheiden, was mit den Zeilen passiert, die keinen Partner finden — und genau das unterscheidet die vier Join-Arten. Die folgenden zwei Tabellen sind so gebaut, dass alle Fälle vorkommen: `A` gibt es nur links, `E` nur rechts, und `D` steht rechts zweimal.

In [16]:
df1 = pd.DataFrame({"key": ["A", "B", "C", "D"], "value": np.random.randn(4)})
df2 = pd.DataFrame({"key": ["B", "D", "D", "E"], "value": np.random.randn(4)})
print(df1, "\n", df2)

  key     value
0   A -0.798653
1   B  2.257592
2   C  1.981691
3   D -0.387363 
   key     value
0   B -0.685048
1   D  0.435486
2   D -1.481497
3   E  0.841617


In [17]:
# SELECT * FROM df1 INNER JOIN df2 ON df1.key = df2.key;
pd.merge(df1, df2, on="key") # inner join ist der default

,key,value_x,value_y
0,B,2.257592,-0.685048
1,D,-0.387363,0.435486
2,D,-0.387363,-1.481497


Der **Inner Join** behält nur Schlüssel, die in beiden Tabellen vorkommen: `B` und `D`. Dass `D` zwei Zeilen liefert, ist der wichtigste Punkt an der ganzen Sache — der Schlüssel ist in `df2` nicht eindeutig, und ein Join vervielfacht dann die Zeilen der anderen Seite. Wer einen Datensatz nach einem Join plötzlich länger vorfindet als vorher, hat fast immer hier das Problem.

In [18]:
# SELECT * FROM df1 LEFT OUTER JOIN df2 ON df1.key = df2.key;
pd.merge(df1, df2, on="key", how="left")

,key,value_x,value_y
0,A,-0.798653,NaN
1,B,2.257592,-0.685048
2,C,1.981691,NaN
3,D,-0.387363,0.435486
4,D,-0.387363,-1.481497


Der **Left Outer Join** behält alle Zeilen der linken Tabelle, auch die ohne Partner: `A` und `C` kommen dazu, mit `NaN` in den Spalten aus `df2`. Das ist der häufigste Fall in der Praxis — man hat einen Datensatz und will ihn um Zusatzinformationen anreichern, ohne dabei Zeilen zu verlieren.

In [19]:
# SELECT * FROM df1 RIGHT OUTER JOIN df2 ON df1.key = df2.key;
pd.merge(df1, df2, on="key", how="right")

,key,value_x,value_y
0,B,2.257592,-0.685048
1,D,-0.387363,0.435486
2,D,-0.387363,-1.481497
3,E,NaN,0.841617


Der **Right Outer Join** ist dasselbe mit vertauschten Rollen: alle Zeilen von `df2`, also auch `E`, und `NaN` für die fehlende linke Seite.

In [20]:
# SELECT * FROM df1 FULL OUTER JOIN df2 ON df1.key = df2.key;
pd.merge(df1, df2, on="key", how="outer")

,key,value_x,value_y
0,A,-0.798653,NaN
1,B,2.257592,-0.685048
2,C,1.981691,NaN
3,D,-0.387363,0.435486
4,D,-0.387363,-1.481497
5,E,NaN,0.841617


Der **Full Outer Join** behält alles, `A` bis `E`, und füllt beide Seiten mit `NaN` auf, wo nichts passt. Danach lohnt sich fast immer ein Blick mit `isna().sum()`, um zu sehen, wie viel Lücke man sich eingehandelt hat.

Daten verändern geht in SQL mit `UPDATE`, in Pandas führen wir einfach Python Code aus, als wenn wir eine einzelne Variable ändern wollten:

In [21]:
# Zustand vor der Änderung:
tips.sort_values("tip").head()

,total_bill,tip,sex,smoker,day,time,size
92,5.75,1.00,Female,Yes,Fri,Dinner,2
111,7.25,1.00,Female,No,Sat,Dinner,1
67,3.07,1.00,Female,Yes,Sat,Dinner,1
236,12.60,1.00,Male,Yes,Sat,Dinner,2
0,16.99,1.01,Female,No,Sun,Dinner,2


In [22]:
# UPDATE tips SET tip = tip*2 WHERE tip < 2;
tips.loc[tips["tip"] < 2, "tip"] *= 2
tips.sort_values("tip").head()

,total_bill,tip,sex,smoker,day,time,size
6,8.77,2.0,Male,No,Sun,Dinner,2
26,13.37,2.0,Male,No,Sat,Dinner,2
27,12.69,2.0,Male,No,Sat,Dinner,2
36,16.31,2.0,Male,No,Sat,Dinner,3
61,13.81,2.0,Male,Yes,Sat,Dinner,2


In [23]:
# Zustand vor der nächsten Änderung:
tips.sort_values("tip", ascending=False).head()

,total_bill,tip,sex,smoker,day,time,size
170,50.81,10.00,Male,Yes,Sat,Dinner,3
212,48.33,9.00,Male,No,Sat,Dinner,4
23,39.42,7.58,Male,No,Sat,Dinner,4
59,48.27,6.73,Male,No,Sat,Dinner,4
141,34.30,6.70,Male,No,Thur,Lunch,6


In [24]:
# DELETE FROM tips WHERE tip > 9;
tips = tips.loc[tips["tip"] <= 9]
tips.sort_values("tip", ascending=False).head()

,total_bill,tip,sex,smoker,day,time,size
212,48.33,9.00,Male,No,Sat,Dinner,4
23,39.42,7.58,Male,No,Sat,Dinner,4
59,48.27,6.73,Male,No,Sat,Dinner,4
141,34.30,6.70,Male,No,Thur,Lunch,6
183,23.17,6.50,Male,Yes,Sun,Dinner,4


## Erster Blick auf einen Datensatz

Pandas bringt die Werkzeuge mit, die wir im Kapitel über deskriptive Statistik besprochen haben und die im Titanic-Notebook noch Handarbeit mit Numpy waren. Das dort 'kaputte' CSV liest `read_csv` direkt, weil es Anführungszeichen versteht. `info` zeigt Spalten, Datentypen und wie viele Werte fehlen: Vergleichen Sie mit dem Titanic-Notebook aus dem Kapitel zu Numpy: Dort haben wir das CSV von Hand in ein TSV umgewandelt, fehlende Werte mit Masken markiert und Gruppen mit Schleifen gezählt.

In [25]:
titanic = pd.read_csv("titanic.csv")
titanic.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


`describe` liefert für jede numerische Spalte Anzahl, Mittelwert, Standardabweichung (mit $n-1$ im Nenner!), Minimum, Quartile und Maximum:

In [26]:
titanic.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


Für kategoriale Spalten zählt `value_counts` die Ausprägungen:

In [27]:
print(titanic["Sex"].value_counts())
print(titanic["Embarked"].value_counts(dropna=False))   # fehlende Werte mitzählen

Sex
male      577
female    314
Name: count, dtype: int64
Embarked
S      644
C      168
Q       77
NaN      2
Name: count, dtype: int64


Fehlende Werte erkennt man mit `isna`. Man kann sie verwerfen (`dropna`) oder ersetzen (`fillna`), zum Beispiel durch den Median. Schiefe und Wölbung aus dem Kapitel über Erwartungswerte sind ebenfalls eingebaut; `kurt` liefert die Exzess-Kurtosis, also die Wölbung minus $3$:

Wie man fehlende Werte, Ausreißer, Textfelder und Datumsangaben systematisch behandelt, ist Thema des Kapitels [Daten putzen](data-cleaning.ipynb).


In [28]:
print(titanic.isna().sum())

alter = titanic["Age"].fillna(titanic["Age"].median())
print("Schiefe:", alter.skew(), " Exzess-Kurtosis:", alter.kurt())

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64
Schiefe: 0.5102446555756495  Exzess-Kurtosis: 0.9938710163801736


Gruppierte Kennzahlen bekommt man wie oben mit `groupby`:

In [29]:
titanic.groupby("Pclass")["Age"].describe()

,count,mean,std,min,25%,50%,75%,max
Pclass,,,,,,,,
1,186.0,38.233441,14.802856,0.92,27.0,37.0,49.0,80.0
2,173.0,29.877630,14.001077,0.67,23.0,29.0,36.0,70.0
3,355.0,25.140620,12.495398,0.42,18.0,24.0,32.0,74.0
